In [1]:
import polars as pl

In [7]:
itemcf_recall_path= '/save/ranker/Valid/itemcf_recall.parquet'
popularity_recall_week1_path= '/save/ranker/Valid/recall_popularity_week1.parquet'
popularity_recall_week2_path= '/save/ranker/Valid/recall_popularity_week2.parquet'
popularity_recall_week3_path= '/save/ranker/Valid/recall_popularity_week3.parquet'
popularity_recall_week4_path= '/save/ranker/Valid/recall_popularity_week4.parquet'
repurchase_recall_path= '/save/ranker/Valid/recall_repurchase.parquet'
w2vec_recall_path= '/save/ranker/Valid/recall_w2vec.parquet'

In [8]:
# 读取召回数据
itemcf_recall = pl.read_parquet(itemcf_recall_path)

popularity_recall_week1 = pl.read_parquet(popularity_recall_week1_path)
popularity_recall_week2 = pl.read_parquet(popularity_recall_week2_path)
popularity_recall_week3 = pl.read_parquet(popularity_recall_week3_path)
popularity_recall_week4 = pl.read_parquet(popularity_recall_week4_path)

repurchase_recall = pl.read_parquet(repurchase_recall_path)

w2vec_recall = pl.read_parquet(w2vec_recall_path)

In [9]:
itemcf_recall.columns,w2vec_recall.columns

(['customer_id', 'article_id', 'score'],
 ['customer_id', 'article_id', 'score'])

In [10]:
itemcf_recall = itemcf_recall.rename({"score": "itemcf_score"})
w2vec_recall = w2vec_recall.rename({"score": "w2v_score"})

In [11]:
itemcf_recall.schema

Schema([('customer_id', String),
        ('article_id', Float64),
        ('itemcf_score', Float64)])

In [12]:
w2vec_recall.schema

Schema([('customer_id', String),
        ('article_id', Int64),
        ('w2v_score', Float64)])

In [13]:
repurchase_recall.schema

Schema([('customer_id', String), ('article_id', Int64), ('rank', UInt8)])

In [14]:
repurchase_recall.schema

Schema([('customer_id', String), ('article_id', Int64), ('rank', UInt8)])

In [15]:
# itemcf召回的结果有误
itemcf_recall = itemcf_recall.with_columns(
    pl.col("article_id").cast(pl.Int64)
)

In [16]:
# 增加用于表示来源的属性
itemcf_recall = itemcf_recall.with_columns([
    pl.lit(1).alias("from_itemcf"),
])

w2vec_recall = w2vec_recall.with_columns([
    pl.lit(1).alias("from_w2vec"),
])

popularity_recall_week1 = popularity_recall_week1.with_columns([
    pl.lit(1).alias("from_popularity_w1"),
])

popularity_recall_week2 = popularity_recall_week2.with_columns([
    pl.lit(1).alias("from_popularity_w2"),
])

popularity_recall_week3 = popularity_recall_week3.with_columns([
    pl.lit(1).alias("from_popularity_w3"),
])

popularity_recall_week4 = popularity_recall_week4.with_columns([
    pl.lit(1).alias("from_popularity_w4"),
])

repurchase_recall = repurchase_recall.with_columns([
    pl.lit(1).alias("from_repurchase"),
])


In [17]:
# 丢去无用的列
popularity_recall_week1=popularity_recall_week1.drop(['rank'])
popularity_recall_week2=popularity_recall_week2.drop(['rank'])
popularity_recall_week3=popularity_recall_week3.drop(['rank'])
popularity_recall_week4=popularity_recall_week4.drop(['rank'])
repurchase_recall=repurchase_recall.drop(['rank'])

In [18]:
# 1. 把所有 recall 放到一个 list 里
dfs = [
    itemcf_recall,
    w2vec_recall,
    popularity_recall_week1,
    popularity_recall_week2,
    popularity_recall_week3,
    popularity_recall_week4,
    repurchase_recall,
]

In [19]:
fill_zero_cols = [
    "itemcf_score",
    "w2v_score",
    "from_itemcf",
    "from_w2vec",
    "from_popularity_w1",
    "from_popularity_w2",
    "from_popularity_w3",
    "from_popularity_w4",
    "from_repurchase",
]
# 合并
data =pl.concat(dfs, how="diagonal") # 默认会将缺失的部分补成null
del dfs

In [ ]:
# 填充缺失值
data=data.with_columns([
        pl.col(c).fill_null(0)
        for c in fill_zero_cols
    ])

In [ ]:
data = (
    data
    .group_by(["customer_id", "article_id"])
    .agg([
        pl.max("itemcf_score").alias("itemcf_score"),
        pl.max("w2v_score").alias("w2v_score"),
        pl.max("from_itemcf").alias("from_itemcf"),
        pl.max("from_w2vec").alias("from_w2vec"),
        pl.max("from_repurchase").alias("from_repurchase"),
        pl.max('from_popularity_w1').alias("from_popularity_w1"),
        pl.max('from_popularity_w2').alias("from_popularity_w2"),
        pl.max('from_popularity_w3').alias("from_popularity_w3"),
        pl.max('from_popularity_w4').alias("from_popularity_w4"),
        pl.count().alias("recall_cnt"),
    ])
    .fill_null(0)
)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# 合并用户和文章属性
articles_path='../data/articles.parquet'
customer_path='../data/customers.parquet'
articles=pl.read_parquet(articles_path)
customers=pl.read_parquet(customer_path)

In [ ]:
data = (
    data
    .join(articles, on="article_id", how="left")
)

In [ ]:
data = (
    data
    .join(customers, on="customer_id", how="left")
)

In [ ]:
data.write_parquet('recall.parquet')

['customer_id',
 'article_id',
 'itemcf_score',
 'w2v_score',
 'from_itemcf',
 'from_w2vec',
 'from_popularity',
 'from_repurchase',
 'product_code',
 'product_type_no',
 'graphical_appearance_no',
 'colour_group_code',
 'perceived_colour_value_id',
 'perceived_colour_master_id',
 'department_no',
 'index_code',
 'index_group_no',
 'section_no',
 'garment_group_no',
 'FN',
 'Active',
 'fashion_news_frequency_Monthly',
 'fashion_news_frequency_NONE',
 'fashion_news_frequency_Regularly',
 'club_member_status_ACTIVE',
 'club_member_status_LEFT CLUB',
 'club_member_status_PRE-CREATE',
 'age_0',
 'age_1',
 'age_2']
